# Part 5 — Data Parallelism and Performance
    Demonstrate Spark partition counts and discuss optimization trade-offs. The provided dataset is intentionally small, so don't claim large-scale speedup from it.

In [ ]:
from pyspark.sql import SparkSession, functions as F
    import os, time
    spark = SparkSession.builder.appName("BankingParallelism").getOrCreate()
    path = "../data/bank.csv"
    if not os.path.exists(path): path = "data/bank.csv"
    df = spark.read.option("header",True).option("inferSchema",True).csv(path)
    print("Rows:", df.count())
    print("Default partitions:", df.rdd.getNumPartitions())

In [ ]:
# Compare partition counts and simple action timings; results depend on hardware and Spark setup.
    for n in [1,2,4,8]:
        part = df.repartition(n, "job")
        start = time.time()
        count = part.groupBy("job").count().count()
        elapsed = time.time() - start
        print({"partitions":n, "group_count":count, "elapsed_seconds":round(elapsed,4)})

## Optimization checklist
    - Filter early and select only needed columns.
    - Avoid repeated actions; cache only when data is reused and memory permits.
    - `repartition()` causes a shuffle; use it when the extra parallelism is worth the cost.
    - `coalesce()` can reduce partitions with less data movement.
    - Check Spark UI for shuffle read/write, task skew, spill, and executor utilization.
    - Partition count should reflect dataset size, file sizes, executor cores, and workload.

    ## Trade-offs
    More partitions can improve concurrency but add scheduling overhead. Too few partitions underuse available cores; too many tiny partitions increase overhead and create many small files. For defensible scalability evidence, benchmark an enlarged dataset on fixed hardware and report the methodology and measured results.

In [ ]:
spark.stop()